# Upstox: check, download NIFTY data, run the backtests
**One-time setup (30 seconds):**
1. Click the **key icon** in the left sidebar (Secrets).
2. Add a secret named exactly `UPSTOX_ANALYTICS_TOKEN` and paste your token as the value. Turn **Notebook access** ON.
3. (Only if your GitHub repo is private) add a second secret `GITHUB_TOKEN` with a read-only GitHub token.
4. Then choose **Runtime > Run all** and wait. Read the output of each cell.

Never paste your token into a cell or into chat. Nothing here can place an order.

In [ ]:
import os
from google.colab import userdata
def secret(k):
    try: return userdata.get(k)
    except Exception: return None
tok = secret('UPSTOX_ANALYTICS_TOKEN')
assert tok, "Add the secret UPSTOX_ANALYTICS_TOKEN (key icon, left sidebar) and turn Notebook access ON, then run again."
os.environ['UPSTOX_ANALYTICS_TOKEN'] = tok
gh = secret('GITHUB_TOKEN')
url = f"https://{gh+'@' if gh else ''}github.com/TechSavvysyedshafai/callable-demo-mcp.git"
!rm -rf repo && git clone -q --branch claude/zerodha-api-live-data-6tf1kn {url} repo
%cd repo
print("token loaded (hidden). code downloaded.")

## Step 1: connection check
You want `[ OK ]` lines. A `[PLUS]` line means the expired-option part needs the Plus plan.

In [ ]:
!python research/upstox/check_setup.py

## Step 2: download about 3 years of 5-minute NIFTY bars + India VIX
Takes a minute or two. The newest 6 months are saved as a hidden final test set.

In [ ]:
!python research/upstox/fetch_upstox.py

## Step 3: NIFTY futures scalping backtest (with real costs, vs random entries)
Look at `verdict`, `net_rs`, `n_trades` and `p_vs_random_entries`. FAIL is the normal result.

In [ ]:
!python research/scalp/evaluate.py --runs 200

## Step 4: NIFTY option-buying backtest (model-based prices from index bars + India VIX)
Indicative only. It prints `realized_vol_pct` vs `mean_iv_pct`; if realized >= implied it marks the run INVALID.

In [ ]:
!python research/scalp/options/evaluate_opt.py --underlying NIFTY --runs 200

## Step 5 (optional, needs Upstox Plus): download real expired-option candles
Upstox keeps only about the last 6 months, so this builds your own archive. Untested against the live API.

In [ ]:
!python research/upstox/download_expired_options.py --underlying NIFTY --expiries 26 --strikes 5

## Step 6: the decisive test - option backtest on REAL expired-option prices
Needs Step 5 to have finished (it can take a while; re-run Step 5 until it says 0 failed). Look at `verdict`, `skipped_signals` and `p_vs_random_entries`.

In [ ]:
!python research/scalp/options/real_backtest.py --runs 200

**When done:** copy the output of steps 1, 3 and 4 and paste it to Claude. It contains no secrets.